# Subprocesses, concurrency and parallelism

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*X. Cid Vidal* (original material, 2025). Adapted by *J. A. Hernando Morata*.

A simulation waits for files, a fit runs on one core while seven others are idle, an
analysis needs an external program. Python can run other programs, overlap the waits, and
use all the cores, if we pick the right tool.

**Objectives**

* run external programs with `subprocess`: output, input, errors and timeouts;
* distinguish **concurrency** from **parallelism**, and **I/O-bound** from **CPU-bound**;
* use **threads**, **processes** and `asyncio`, and **measure** what each one gains;
* understand the **GIL** and why processes need their workers in a module;
* use `concurrent.futures`, the modern interface to threads and processes.

## 1. Subprocesses

The `subprocess` module runs another program as a **child process**: it can pass it input,
capture its output and check how it ended. It is the bridge between Python and the shell.

`subprocess.run` runs a command, **waits** for it to finish and returns a
`CompletedProcess`:

In [ ]:
import subprocess
import sys

HELLO = [sys.executable, '-c', 'print("Hello from a subprocess")']

result = subprocess.run(HELLO, capture_output=True, text=True)
print(type(result).__name__)
print(repr(result.stdout), result.returncode)

* the command is a **list**: program and arguments, no shell parsing;
* `sys.executable` is the Python we are running: a child that exists on every OS;
* `capture_output=True` captures `stdout` and `stderr` (otherwise they go to the console);
* `text=True` gives `str` instead of `bytes`;
* `returncode` is the exit status: `0` means success.

### 1.1 Working directory

`cwd` sets the directory where the command runs. We list a temporary directory, so that
the output does not depend on your machine (on Windows, `['cmd', '/c', 'dir']`):

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    for name in ['data.txt', 'notes.md', 'run.txt']:
        (Path(tmp) / name).touch()
    result = subprocess.run(['ls'], cwd=tmp, capture_output=True, text=True)

print(result.stdout.split())

### 1.2 Passing input

`input=` is sent to the **standard input** of the child. Our child is a tiny calculator
(the role of the Unix `bc`): again `sys.executable`, with a one-line program given by
`-c`.

In [ ]:
CALC = [sys.executable, '-c', 'import sys; print(eval(sys.stdin.read()))']

result = subprocess.run(CALC, input='5 + 10', capture_output=True, text=True)
print(result.stdout)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — What comes back?</b></p>

**Decide** what this prints, before running it.

```python
results = [subprocess.run(CALC, input=expr, capture_output=True, text=True)
           for expr in ['5 + 510 * 2', '2 ** 10']]
print([r.stdout for r in results])
```

</div>

In [ ]:
results = [subprocess.run(CALC, input=expr, capture_output=True, text=True)
           for expr in ['5 + 510 * 2', '2 ** 10']]
print([r.stdout for r in results])

<details>
<summary><b>Solution</b></summary>

`['1025\n', '1024\n']`. The child **prints**, so we get text, not numbers, and with the
final newline. Use `int(r.stdout)` or `r.stdout.strip()`. (`eval` runs any code: use it
only with input you trust.)

</details>

### 1.3 Errors and timeouts

* `check=True` raises `CalledProcessError` if the exit status is not `0`;
* `timeout=` (seconds) kills the child and raises `TimeoutExpired`.

Two children: one fails at once with exit status 1, the other sleeps longer than the
timeout.

In [ ]:
FAIL = [sys.executable, '-c', 'import sys; sys.exit("no such file")']
SLOW = [sys.executable, '-c', 'import time; time.sleep(10)']

for command in [FAIL, SLOW]:
    try:
        subprocess.run(command, capture_output=True, text=True, timeout=2, check=True)
    except subprocess.CalledProcessError as e:
        print('failed with status', e.returncode, ':', e.stderr.strip())
    except subprocess.TimeoutExpired as e:
        print('timed out after', e.timeout, 's')

### 1.4 `Popen`: do not wait

* `run` blocks until the child ends;
* `Popen` **starts** the child and returns at once: the parent keeps working;
* `communicate()` sends the input, **waits** for the end and returns all the output;
* a one-shot exchange, not a dialogue (for that, read and write `process.stdout` and
  `process.stdin` line by line).

In [ ]:
process = subprocess.Popen(CALC, stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)

print('the child is running, the parent is free')       # do other work here
output, error = process.communicate(input='5 * 5')
print('output:', output.strip(), '| status:', process.returncode)

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A directory from the shell</b></p>

Unix/macOS commands (on Windows, use WSL or Git Bash). In a
`tempfile.TemporaryDirectory()`, and using **only** `subprocess` for the file system:

1. create a directory `test_subprocess_dir` (`mkdir`);
2. create inside it `file1.txt`, `file2.txt`, `file3.txt` and `notes.md` (`touch`);
3. list it with `ls`, and print only the names that end in `.txt`.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
with tempfile.TemporaryDirectory() as tmp:
    folder = Path(tmp) / 'test_subprocess_dir'
    subprocess.run(['mkdir', '-p', str(folder)], check=True)
    names = ['file1.txt', 'file2.txt', 'file3.txt', 'notes.md']
    subprocess.run(['touch', *names], cwd=folder, check=True)
    listing = subprocess.run(['ls'], cwd=folder, capture_output=True, text=True, check=True)

print([name for name in listing.stdout.split() if name.endswith('.txt')])
```

The filter is done in Python, not with `ls *.txt`: without a shell, `*` is not expanded.
`shell=True` would expand it, at the price of shell-injection risks.

</details>

## 2. Concurrency and parallelism

* **Concurrency**: several tasks **in progress** in overlapping time periods. One worker
  can do it, switching to another task while the first one waits.
* **Parallelism**: several tasks **executing at the same instant**, on several cores.

```
               time -->
sequential   A ##....
             B       ##....    one worker: B starts when A ends
concurrent   A ##....
             B   ##....        one worker: B runs while A waits
parallel     A ##....
             B ##....          two workers: A and B run at once

# running (uses the CPU)       . waiting (I/O)
```

What a task spends its time on decides which tool helps:

* **I/O-bound**: waiting for disk, network, another program, a `sleep`. Concurrency
  helps: overlap the waits.
* **CPU-bound**: computing. Only parallelism helps: more cores.

| Tool | Runs on | Good for |
|:--|:--|:--|
| `threading` | several threads of one process (shared memory) | I/O-bound |
| `multiprocessing` | several processes (separate memory) | CPU-bound |
| `asyncio` | one thread, tasks that `await` | many I/O-bound tasks |

### 2.1 The GIL

In CPython, the **Global Interpreter Lock** lets only **one thread at a time** execute
Python code. Hence:

* threads do **not** run pure-Python computations in parallel;
* but a thread **releases** the GIL while it waits for I/O, and inside many C extensions
  (NumPy, for instance), so threads do help there;
* processes have one interpreter, and one GIL, each: true parallelism;
* `asyncio` does **not** bypass the GIL: it is a single thread that avoids idle waiting.

We will **measure** each of these claims, with `time.perf_counter()`.

***[+] Lookout.*** CPython has an optional **free-threaded** build without the GIL: experimental in 3.13,
officially supported from 3.14 ([PEP 703](https://peps.python.org/pep-0703/),
[PEP 779](https://peps.python.org/pep-0779/)), installed as `python3.13t` or
`python3.14t`. `sys._is_gil_enabled()` tells which one you run. What this notebook says
about threads holds **with the GIL**; many libraries are not ready for free threading yet.

## 3. Threads

A `threading.Thread` runs a function (`target`) with its arguments (`args`). `start()`
launches it and returns at once; `join()` waits for it to end.

Two tasks that mostly **wait**, as if reading from an instrument. Each one notes when it
produced each item:

In [ ]:
import threading
import time

def produce(items, delay, log, t0):
    for item in items:
        log.append((round(time.perf_counter() - t0, 1), item))
        time.sleep(delay)           # waiting: I/O-bound

In [ ]:
log, t0 = [], time.perf_counter()
numbers = threading.Thread(target=produce, args=('12345', 0.2, log, t0))
letters = threading.Thread(target=produce, args=('ABCDE', 0.3, log, t0))

numbers.start(); letters.start()
numbers.join(); letters.join()

print(f'total {time.perf_counter() - t0:.1f} s')
print(' '.join(f'{t}:{item}' for t, item in sorted(log)))     # time:item

The two tasks ran **concurrently**: the total is the longest task (≈ 1.5 s), not the sum
(≈ 2.5 s), and the items are interleaved in time. We collect the results in a list and
print **after** `join()`: threads that print by themselves mix their lines unpredictably.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Forget the join</b></p>

**Decide** what happens if we remove `numbers.join(); letters.join()` from the cell
above. What does it print? And what if the next cell prints something?

</div>

<details>
<summary><b>Solution</b></summary>

`start()` returns at once, so the cell goes on while the threads still run: the total is
≈ 0 s and the log holds at most the first items. The threads go on
in the background, and anything they print appears in whatever cell runs next. Always
`join` the threads you start (or use an executor, next section, which does it for you).

</details>

### 3.1 Shared memory

Threads share the memory of their process: `log` above is the **same** list for both.
That is convenient, and dangerous: if two threads update the same object at the same
time, one update can be lost (a **race condition**: rare with the GIL, but real, and more
common on free-threaded builds). `threading.Lock` makes a block exclusive:

```python
lock = threading.Lock()
with lock:
    counter += 1
```

The safest rule: do not share mutable state; let each task **return** its result.

## 4. `concurrent.futures`

The modern interface
([docs](https://docs.python.org/3/library/concurrent.futures.html)): an **executor** keeps
a pool of workers, `map` sends them the tasks and returns the **results** in order, and
the `with` block joins them at the end.

Four tasks that wait 0.5 s each:

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def wait(seconds):
    time.sleep(seconds)
    return seconds

t0 = time.perf_counter()
results = [wait(0.5) for _ in range(4)]
print(f'sequential {time.perf_counter() - t0:.1f} s', results)

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(wait, [0.5] * 4))
print(f'threads    {time.perf_counter() - t0:.1f} s', results)

### 4.1 CPU-bound work with threads

Now a task that **computes**: count the primes below $n$ in pure Python. It lives in a
small module, `fca_workers.py`, next to this notebook (section 5 explains why):

In [ ]:
import inspect
import fca_workers

print(inspect.getsource(fca_workers.count_primes))

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Four threads, four cores</b></p>

One call `count_primes(N)` takes a time $T$ on your machine. **Predict** how long four
calls take sequentially, and with a `ThreadPoolExecutor` of four threads.

</div>

In [ ]:
N, tasks = 250_000, 4
print('GIL enabled:', getattr(sys, '_is_gil_enabled', lambda: True)())

t0 = time.perf_counter()
results = [fca_workers.count_primes(N) for _ in range(tasks)]
t_seq = time.perf_counter() - t0
print(f'sequential {t_seq:.2f} s', results)

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=tasks) as pool:
    results = list(pool.map(fca_workers.count_primes, [N] * tasks))
t_thr = time.perf_counter() - t0
print(f'threads    {t_thr:.2f} s', results, f'speedup {t_seq / t_thr:.1f}')

<details>
<summary><b>Solution</b></summary>

About $4T$ both ways: with the GIL, the threads give **no speedup** (≈ 1, sometimes a
small slowdown). The GIL lets one thread at a time execute Python code, so the four
threads take turns on one core. Your times will differ from the ones printed in the book;
the speedup will be close to 1 on any machine with the GIL (on a free-threaded build it
would approach the number of cores).

</details>

## 5. Processes

Each process has its own interpreter, memory and GIL: they run in parallel on different
cores. The price: starting a process is slow (tens of ms), and the arguments and results
travel between processes **serialised** with `pickle`.

`ProcessPoolExecutor` has the same interface as the thread pool. `mp_context` chooses how
the children start (section 5.1): `spawn` behaves the same on Linux, macOS and Windows.

In [ ]:
import multiprocessing
import os
from concurrent.futures import ProcessPoolExecutor

SPAWN = multiprocessing.get_context('spawn')
print('cores:', os.process_cpu_count())

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=tasks, mp_context=SPAWN) as pool:
    results = list(pool.map(fca_workers.count_primes, [N] * tasks))
t_proc = time.perf_counter() - t0
print(f'processes  {t_proc:.2f} s', results)
print(f'speedup: threads {t_seq / t_thr:.1f}, processes {t_seq / t_proc:.1f}')

### 5.1 Why a module?

A child process must find the function it runs. How, depends on the **start method**
([docs](https://docs.python.org/3/library/multiprocessing.html#contexts-and-start-methods)):

* `spawn` (default on Windows and macOS): a fresh interpreter that **imports** the
  function by its module and name;
* `fork` (default on Linux up to 3.13): a copy of the parent, which hides the problem; it
  may deadlock if the parent has threads (Python warns), and 3.14 moves Linux to
  `forkserver`, which imports like `spawn`;
* a function defined in a notebook cell lives in `__main__`, which the child cannot import:
  `AttributeError: Can't get attribute ...`, and the pool breaks (`BrokenProcessPool`).

In [ ]:
print('default start method here:', multiprocessing.get_start_method())

* put the workers in a `.py` module and import it: as we did, and portable;
* fix the start method with `mp_context`, as we did, so the code behaves the same on every
  OS;
* or use the package `multiprocess` (same API, serialises with `dill`).

### 5.2 In a script

In a script, the module that `spawn` imports in the child is the script itself. Without
the guard, the child would try to start its own processes while it is still starting up,
and Python stops it with `RuntimeError: An attempt has been made to start a new process
before the current process has finished its bootstrapping phase`.

```python
# primes.py
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from fca_workers import count_primes

if __name__ == '__main__':          # only in the parent, not on import
    spawn = multiprocessing.get_context('spawn')
    with ProcessPoolExecutor(mp_context=spawn) as pool:
        print(list(pool.map(count_primes, [250_000] * 4)))
```

The lower-level `multiprocessing.Process(target=..., args=...)` has `start` and `join`
like a thread; `multiprocessing.Pool` is the older version of the executor.

***[+] Lookout.*** See the cores working. In a terminal, save this as `burn.py`, run it, and watch the CPU
monitor (`htop`, Activity Monitor on a Mac, Task Manager on Windows): one core at 100 %
per process, for 10 s.

```python
import multiprocessing as mp, time

def burn():
    while True:
        pass

if __name__ == '__main__':
    processes = [mp.Process(target=burn) for _ in range(4)]
    for p in processes: p.start()
    time.sleep(10)
    for p in processes: p.kill()
```

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — More processes, more speed?</b></p>

In the process-pool cell, what do you expect if `tasks = 4` and `max_workers=16` on a
machine with 8 cores? And with 32 tasks? And with 400 tasks of `count_primes(1000)` each?
When are processes **slower** than a plain loop?

</div>

<details>
<summary><b>Solution</b></summary>

* 4 tasks, 16 workers: with `spawn` the pool starts processes **on demand**, so 4 tasks use
  at most 4 processes: no penalty, no gain. With `fork` all 16 start at once, and we pay
  their start-up.
* More workers than cores help only if there are more tasks than cores, and only up to the
  number of cores: with 32 tasks, 8 workers already use the 8 cores; 16 just take turns.
* 400 tiny tasks: each one is shorter than the cost of sending it to another process and
  getting the result back, so a plain loop wins.

Processes pay off for few, **large**, independent tasks (`map` has a `chunksize` argument
to send the small ones in batches).

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Two computations in parallel</b></p>

With a `ProcessPoolExecutor`, compute **at the same time** `fca_workers.fibonacci(32)` and
`fca_workers.count_primes(300_000)`. Use `pool.submit(f, *args)`, which returns a
**future**, and its `.result()`. Compare the time with running both sequentially.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
t0 = time.perf_counter()
sequential = fca_workers.fibonacci(32), fca_workers.count_primes(300_000)
print(f'sequential {time.perf_counter() - t0:.2f} s', sequential)

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2, mp_context=SPAWN) as pool:
    fib = pool.submit(fca_workers.fibonacci, 32)
    primes = pool.submit(fca_workers.count_primes, 300_000)
    parallel = fib.result(), primes.result()
print(f'processes  {time.perf_counter() - t0:.2f} s', parallel)
```

The parallel time is about the **longest** of the two tasks, plus the start-up of the
processes; the sequential one is their **sum**. `submit` runs different functions;
`map` runs one function over many arguments.

</details>

## 6. `asyncio`

Concurrency in **one thread**. A **coroutine** (`async def`) runs until an `await`, where
it hands control back to the **event loop**, which runs another coroutine meanwhile.
`asyncio.gather` runs several of them concurrently.

Three simulated downloads, of 1.0, 0.5 and 1.5 s:

In [ ]:
import asyncio

async def download(name, delay):
    await asyncio.sleep(delay)          # a network wait
    return f'{name}: {delay} s'

t0 = time.perf_counter()
results = await asyncio.gather(download('A', 1.0), download('B', 0.5), download('C', 1.5))
print(f'total {time.perf_counter() - t0:.1f} s', results)

* the total is the longest download, ≈ 1.5 s; `gather` returns the results in order;
* the notebook already runs an event loop, so we can `await` at top level. In a script,
  write an `async def main()` and run it with `asyncio.run(main())`;
* the switch happens **only** at an `await`: a coroutine that never awaits blocks
  everything.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — A blocking call</b></p>

**Predict** the total time if `download` uses `time.sleep(delay)` instead of
`await asyncio.sleep(delay)`.

</div>

In [ ]:
async def download_blocking(name, delay):
    time.sleep(delay)                   # blocks the only thread
    return f'{name}: {delay} s'

t0 = time.perf_counter()
results = await asyncio.gather(download_blocking('A', 1.0), download_blocking('B', 0.5),
                               download_blocking('C', 1.5))
print(f'total {time.perf_counter() - t0:.1f} s', results)

<details>
<summary><b>Solution</b></summary>

≈ 3.0 s, the **sum**: `time.sleep` never gives control back to the event loop, so the
downloads run one after the other. Move blocking calls out of the loop:

* blocking I/O (`time.sleep`, `requests.get`, reading a file):
  `await asyncio.to_thread(f, *args)` runs it in a thread and keeps the loop responsive;
* a heavy computation: `to_thread` still keeps the loop responsive, but with the GIL it
  does **not** run in parallel; use a process pool,
  `await asyncio.get_running_loop().run_in_executor(pool, f, *args)` with
  `pool = ProcessPoolExecutor(mp_context=SPAWN)`.

`asyncio` has no native asynchronous file I/O (the package `aiofiles` provides it).

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Several programs at once</b></p>

Run four children `[sys.executable, '-c', 'import time; time.sleep(0.5); print(42)']`
**concurrently** and collect their outputs, in under 1 s. Do you need processes for this,
or are threads enough? Why?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
SLEEPER = [sys.executable, '-c', 'import time; time.sleep(0.5); print(42)']

def run(command):
    return subprocess.run(command, capture_output=True, text=True).stdout.strip()

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    outputs = list(pool.map(run, [SLEEPER] * 4))
print(f'{time.perf_counter() - t0:.1f} s', outputs)
```

Threads are enough: each child is **already** a separate process, and the thread that
launched it only **waits** for it (I/O-bound, the GIL is released). `Popen` without
threads works too: start the four, then `communicate` with each.

</details>

## 7. Summary

* **I/O-bound** (waiting): overlap the waits, with threads or `asyncio`;
* **CPU-bound** (computing): use several cores, with processes (workers in a module,
  `spawn`);
* **other programs**: `subprocess.run` (wait) or `Popen` (do not wait), with `check=` and
  `timeout=`;
* always **measure**: the speedup, not the intuition, decides.

<details>
<summary><b>[i] Cheat sheet — concurrency in Python</b></summary>

| | `asyncio` | `threading` | `multiprocessing` |
|:--|:--|:--|:--|
| Runs on | one thread, cooperative | OS threads, shared memory | processes, separate memory |
| Who switches tasks | the program (`await`) | the OS | the OS |
| Bypasses the GIL | no | only in I/O and C code | yes |
| Uses several cores | no | not for pure Python | yes |
| Best for | many I/O-bound tasks | blocking I/O | CPU-bound tasks |
| Cost | blocking calls freeze the loop | race conditions | start-up, pickling |
| Interface | `async`/`await`, `gather` | `ThreadPoolExecutor` | `ProcessPoolExecutor` |

`subprocess`: `run` (wait), `Popen` (do not wait), `check=`, `timeout=`, `input=`, `cwd=`.

Docs: [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html),
[start methods](https://docs.python.org/3/library/multiprocessing.html#contexts-and-start-methods),
free threading [PEP 703](https://peps.python.org/pep-0703/),
[PEP 779](https://peps.python.org/pep-0779/).

</details>

A restaurant with many tables:

* `asyncio`: **one** very efficient waiter, who serves another table while the kitchen
  cooks;
* `threading`: **several** waiters sharing one kitchen, who must not grab the same plate;
  and only one of them can cook at a time (the GIL);
* `multiprocessing`: **several** small restaurants, each with its own kitchen: they cook in
  parallel, but passing dishes between them is slow.

### Check yourself

1. What is the difference between `subprocess.run` and `subprocess.Popen`?
2. Why does `subprocess.run(['ls', '*.txt'])` not list the `.txt` files?
3. Concurrency or parallelism: what does a single-core machine allow?
4. Four threads computing in pure Python on four cores, with the GIL: what speedup? And
   waiting for the network?
5. Why must the worker of a `ProcessPoolExecutor` with `spawn` be in a module?
6. What happens to the other coroutines when one calls `time.sleep(1)`?

<details>
<summary><b>Solutions</b></summary>

1. `run` waits for the child and returns its result; `Popen` starts it and returns at once.
2. Without a shell nobody expands `*`: `ls` looks for a file literally called `*.txt`.
3. Concurrency only (tasks interleaved); parallelism needs several cores.
4. None: the GIL lets one thread at a time run Python code. Waiting: close to 4, the GIL
   is released while waiting.
5. With `spawn` (default on macOS and Windows) the child imports the function, and a
   function defined in the notebook (`__main__`) cannot be imported.
6. They are frozen for 1 s: the event loop runs in that same thread and only switches at
   an `await`.

</details>